In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score, classification_report

In [4]:
df = pd.read_parquet("processed_ml_dataset.parquet")
data = pd.read_parquet("D:/TrendSpotting/TrendSpotting/download_dataset/data/openalex_corpus_1m.parquet")
df.head()

,doc_id,source_tier,novelty_raw,cluster_density,is_outlier_cluster,ppmi_domain_score,commercial_maturity_index,citation_velocity,citation_acceleration,distinct_venues_ratio,author_growth_rate,has_ref_data,target_emergence
0,W2280936323,journal,1.0,0.0,0,0.0,0.0,0.0,1.0,0.0,0.0,1,<NA>
1,W4232448486,other,1.0,0.0,0,0.0,NaN,0.0,1.0,0.0,0.0,1,<NA>
2,W2099050061,journal,1.0,0.0,0,0.0,NaN,0.0,1.0,0.0,0.0,1,<NA>
3,W2152433346,journal,1.0,0.0,0,0.0,0.0,0.0,1.0,0.0,0.0,1,<NA>
4,W1718665810,journal,1.0,0.0,0,0.0,0.0,0.0,1.0,0.0,0.0,1,<NA>


In [5]:
df = df.dropna(subset=["target_emergence"])
df.head()

,doc_id,source_tier,novelty_raw,cluster_density,is_outlier_cluster,ppmi_domain_score,commercial_maturity_index,citation_velocity,citation_acceleration,distinct_venues_ratio,author_growth_rate,has_ref_data,target_emergence
149311,W1970115948,journal,0.729391,0.086066,0,0.0,0.0,0.0,1.0,0.0,0.303682,1,0
149312,W2188122077,journal,0.786411,0.080600,0,0.0,0.0,0.0,1.0,0.0,-0.994623,1,0
149313,W2096533605,other,0.727593,0.101977,0,0.0,NaN,0.0,1.0,0.0,-0.125163,1,0
149314,W2066933239,journal,0.834350,0.128892,0,0.0,0.0,0.0,1.0,0.0,-0.719123,1,0
149315,W1536739630,other,0.685130,0.095773,0,0.0,NaN,0.0,1.0,0.0,0.174353,1,0


In [6]:
df["commercial_maturity_index"].isna().sum()

np.int64(139016)

In [7]:
df[df["commercial_maturity_index"].isna()][
    ["doc_id", "commercial_maturity_index"]
].head(20)

,doc_id,commercial_maturity_index
149313,W2096533605,NaN
149315,W1536739630,NaN
149323,W2144187926,NaN
149330,W1614633289,NaN
149335,W4231123991,NaN
149336,W2182122764,NaN
149342,W834014647,NaN
149347,W2799752344,NaN
149348,W2556095466,NaN
149354,W2037504719,NaN


In [8]:
print(df["target_emergence"].value_counts())
print(df["target_emergence"].value_counts(normalize=True))

target_emergence
0    545598
1     42638
Name: count, dtype: Int64
target_emergence
0    0.927515
1    0.072485
Name: proportion, dtype: Float64


In [9]:
X = df.drop(["target_emergence", "doc_id"], axis=1)
y = df.target_emergence

In [10]:
X.head()

,source_tier,novelty_raw,cluster_density,is_outlier_cluster,ppmi_domain_score,commercial_maturity_index,citation_velocity,citation_acceleration,distinct_venues_ratio,author_growth_rate,has_ref_data
149311,journal,0.729391,0.086066,0,0.0,0.0,0.0,1.0,0.0,0.303682,1
149312,journal,0.786411,0.080600,0,0.0,0.0,0.0,1.0,0.0,-0.994623,1
149313,other,0.727593,0.101977,0,0.0,NaN,0.0,1.0,0.0,-0.125163,1
149314,journal,0.834350,0.128892,0,0.0,0.0,0.0,1.0,0.0,-0.719123,1
149315,other,0.685130,0.095773,0,0.0,NaN,0.0,1.0,0.0,0.174353,1


In [11]:
y.head()

149311    0
149312    0
149313    0
149314    0
149315    0
Name: target_emergence, dtype: Int64

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [13]:
model = CatBoostClassifier(
    iterations=3000,
    depth=10,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="F1",
    auto_class_weights="Balanced",
    random_seed=42,
    verbose=100
)

In [14]:
model.fit(
    X_train,
    y_train,
    cat_features=["source_tier"],
    eval_set=(X_test, y_test),
    early_stopping_rounds=100
)

0:	learn: 0.6334717	test: 0.6327376	best: 0.6327376 (0)	total: 268ms	remaining: 13m 22s
100:	learn: 0.6969670	test: 0.6946674	best: 0.6947006 (99)	total: 12.9s	remaining: 6m 11s
200:	learn: 0.7169780	test: 0.7090234	best: 0.7090234 (200)	total: 25.4s	remaining: 5m 53s
300:	learn: 0.7467060	test: 0.7361403	best: 0.7361403 (300)	total: 37.8s	remaining: 5m 38s
400:	learn: 0.7829222	test: 0.7723259	best: 0.7723259 (400)	total: 51.8s	remaining: 5m 35s
500:	learn: 0.8138944	test: 0.8014080	best: 0.8014080 (500)	total: 1m 6s	remaining: 5m 29s
600:	learn: 0.8378882	test: 0.8234408	best: 0.8234408 (600)	total: 1m 19s	remaining: 5m 19s
700:	learn: 0.8563418	test: 0.8395661	best: 0.8395661 (700)	total: 1m 40s	remaining: 5m 28s
800:	learn: 0.8704063	test: 0.8532684	best: 0.8532684 (800)	total: 2m	remaining: 5m 30s
900:	learn: 0.8810721	test: 0.8648525	best: 0.8648525 (900)	total: 2m 20s	remaining: 5m 28s
1000:	learn: 0.8920446	test: 0.8745059	best: 0.8745059 (1000)	total: 2m 39s	remaining: 5m 17s


CatBoostClassifier(auto_class_weights='Balanced', depth=10, eval_metric='F1', iterations=3000, learning_rate=0.05, loss_function='Logloss', random_seed=42, verbose=100)

In [15]:
# Первый тест

pred = model.predict(X_test).astype(int).ravel()
proba = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, pred))
print("ROC-AUC:", roc_auc_score(y_test, proba))

              precision    recall  f1-score   support

         0.0       1.00      0.93      0.96    109120
         1.0       0.51      0.95      0.66      8528

    accuracy                           0.93    117648
   macro avg       0.75      0.94      0.81    117648
weighted avg       0.96      0.93      0.94    117648

ROC-AUC: 0.978153174504857


In [16]:
from catboost import Pool
import numpy as np
import pandas as pd

test_pool = Pool(
    X_test,
    cat_features=["source_tier"]
)

shap_values = model.get_feature_importance(
    test_pool,
    type="ShapValues"
)

# Последний столбец — базовое значение модели,
# поэтому его убираем
shap_features = shap_values[:, :-1]

In [17]:
shap_importance = pd.DataFrame({
    "feature": X_test.columns,
    "SHAP_importance": np.abs(shap_features).mean(axis=0)
})

shap_importance = shap_importance.sort_values(
    "SHAP_importance",
    ascending=False
)

print(shap_importance)

                      feature  SHAP_importance
2             cluster_density         0.914127
9          author_growth_rate         0.878940
4           ppmi_domain_score         0.190876
1                 novelty_raw         0.185758
0                 source_tier         0.101375
5   commercial_maturity_index         0.052461
7       citation_acceleration         0.041243
6           citation_velocity         0.039497
8       distinct_venues_ratio         0.014104
3          is_outlier_cluster         0.002672
10               has_ref_data         0.000000


In [18]:
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

# Вероятности класса 1
proba = model.predict_proba(X_test)[:, 1]

thresholds = [0.85, 0.90, 0.92, 0.95, 0.9787982]

results = []

for threshold in thresholds:
    pred = (proba >= threshold).astype(int)

    results.append({
        "Threshold": threshold,
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred)
    })

results_df = pd.DataFrame(results)

print(results_df)

   Threshold  Precision    Recall        F1
0   0.850000   0.790063  0.546318  0.645962
1   0.900000   0.856792  0.368316  0.515171
2   0.920000   0.886806  0.288462  0.435321
3   0.950000   0.934813  0.158068  0.270411
4   0.978798   0.982143  0.064493  0.121039


In [19]:
model.save_model("catboost_model.cbm")